# Training the book segmenter the robot uses

The robot is the book-return cart (`bila/control/cart_demo.py`): a Panda on a lift whose only camera is on its
wrist. Every time it looks at the row (the first look's three views along the row, and a look from in front of
the slot after each step), the camera pipeline (`bila/vision/rgbd_perception.py`) needs **one pixel mask per
book** in the colour image. It fits each spine to the depth pixels inside that book's mask, and reads the
call-number sticker on it to look the book up in the catalogue.

By default the masks are a cheat: `OracleSegmenter` reads them off the simulator's object ids. This notebook
trains the model that replaces it, `LearnedSegmenter` in `bila/vision/learned_segmenter.py`: a Mask R-CNN
(torchvision, starting from COCO weights) with a single class, "book". The robot then runs with
`--segmenter ~/bila_data/segmenter/segmenter.pt`.

1. **Make training data** (`bila/vision/make_vision_dataset.py`): random scenes rendered from the arm poses the
   robot really uses. The simulator labels every pixel, so no hand-labelling is needed.
2. **Look at it**: colour image, the per-book masks it will learn, and the depth.
3. **Train** (`bila/vision/learned_segmenter.py train`).
4. **Score it** against the simulator's true masks, on the views the robot uses.
5. **See its masks** next to the truth.
6. **Check it on the robot**: cart episodes with the learned masks against the simulator's.

**Why masks and not YOLO boxes?** PBBM detected spines with YOLOv5 *boxes*, which are axis-aligned, so they
had to warp the image until spines stood upright. Here each spine is fitted to the depth pixels *inside its
mask*. A box around a leaning book also takes in its neighbours' pixels.

**Time on an Apple M4 Pro (MPS)**: data about 1.3 s an image on 3 workers; training about 2.5 s a step (2 images).
The defaults below are a first run of about an hour; the README's full recipe is in the comments.


In [ ]:
import os, sys, json, subprocess, pathlib
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")    # before torch is imported: CPU for any op MPS lacks
ROOT = pathlib.Path.cwd().parent                               # the repo: the bila package is there
sys.path.insert(0, str(ROOT))

DATA = pathlib.Path("~/bila_data").expanduser()                # where the README keeps data and weights
TRAIN, VAL, OUT = DATA / "train", DATA / "val", DATA / "segmenter"

TRAIN_SCENES = 150      # README: 670 (about 2900 images)
VAL_SCENES = 20         # README: 70
WORKERS = 6             # scenes rendered in parallel
EPOCHS = 3              # README: 8
MAX_STEPS = None        # e.g. 20 for a quick check of the whole notebook

## 1. Make the training data

Each scene is one random bookcase, seen through the wrist camera only, from real arm poses:
- **cart scenes** (40%): the robot's own scene: the repo's real books with call-number stickers, the cart beside
  the shelf. Views: the first look from in front of the slot, close-ups of the row near the slot and of the tray.
- **shelf scenes** (60%): all five scenarios, half plain boxes and half real covers, from the first look's three
  views and a random one. More variety than the cart alone, so the model learns "book", not "these books".

Every shelf board carries the ArUco markers the robot uses as its reference (`shelf_scene.fiducials`); they are
part of the shelf, not books, so they are never in a mask. Lighting and colours vary, and the depth carries a
D405's errors. Each image gets the colour image, depth, a 16-bit mask (0 = no book, k = k-th book) and a JSON label.

Seeds: training from 2000, validation from 9000 (0-49 are the controller tests, 1000-1999 `eval_perception`'s).

It runs as a separate process: MuJoCo rendering in a process that has imported torch crashes. It can be
stopped and rerun: scenes already done are skipped. **Data made before the cart lost its fixed cameras and the
shelves got their markers** no longer looks like what the robot sees: the next cell stops if it finds any; delete
those folders and make them again.


In [ ]:
def run(*args):
    # Run a repo script with this kernel's Python, printing its output as it comes.
    p = subprocess.Popen([sys.executable, *map(str, args)], cwd=ROOT, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        if "ARB_clip_control" not in line:                     # a harmless MuJoCo warning on macOS
            print(line, end="")
    if p.wait():
        raise RuntimeError(f"{args[0]} failed")

def stale(folder):
    # Made for the old cart: images from its fixed cameras (row_cam, tray_cam), shelves without markers.
    index = folder / "index.jsonl"
    return index.exists() and any(json.loads(l)["view"] in ("row_cam", "tray_cam") for l in open(index))

for folder in (TRAIN, VAL):
    if stale(folder):
        raise RuntimeError(f"{folder} was made for the old cart (row_cam/tray_cam): delete it and rerun this cell")

run("-m", "bila.vision.make_vision_dataset", "--out", TRAIN, "--first-seed", 2000, "--scenes", TRAIN_SCENES, "--workers", WORKERS)
run("-m", "bila.vision.make_vision_dataset", "--out", VAL, "--first-seed", 9000, "--scenes", VAL_SCENES, "--workers", WORKERS)
print(len(list((TRAIN / "rgb").glob("*.png"))), "training images,", len(list((VAL / "rgb").glob("*.png"))), "validation images")


## 2. What the model learns from

Left: the colour image, the model's only input. Middle: the target, one colour per book. Right: depth, which
the model never sees; the spine fit uses it afterwards. `view` says which camera and pose took the image.

In [ ]:
import cv2, numpy as np, matplotlib.pyplot as plt

def colour_masks(mask, seed=0):
    # A random colour per book id; black where there is no book.
    lut = np.random.default_rng(seed).integers(40, 255, size=(int(mask.max()) + 1, 3)).astype(np.uint8)
    lut[0] = 0
    return lut[mask]

stems = sorted(p.stem for p in (TRAIN / "labels").glob("*.json"))
for stem in stems[:: max(1, len(stems) // 3)][:3]:
    rgb = cv2.cvtColor(cv2.imread(str(TRAIN / "rgb" / f"{stem}.png")), cv2.COLOR_BGR2RGB)
    mask = cv2.imread(str(TRAIN / "masks" / f"{stem}.png"), cv2.IMREAD_UNCHANGED).astype(np.int32)
    depth = cv2.imread(str(TRAIN / "depth" / f"{stem}.png"), cv2.IMREAD_UNCHANGED).astype(float) / 1e4   # 0.1 mm units
    label = json.load(open(TRAIN / "labels" / f"{stem}.json"))
    fig, ax = plt.subplots(1, 3, figsize=(16, 3.2))
    ax[0].imshow(rgb); ax[1].imshow(colour_masks(mask)); ax[2].imshow(np.where(depth > 0, depth, np.nan))
    for a, t in zip(ax, ("colour (input)", f"masks: {mask.max()} books (target)", "depth (not an input)")):
        a.set_title(t, fontsize=9); a.axis("off")
    fig.suptitle(f"{stem}: view {label['view']}", fontsize=10)
    plt.show()

## 3. Train

`learned_segmenter.train`: SGD with a short warm-up, then a cosine decay; flips and colour jitter as
augmentation. It saves `segmenter.pt` after every epoch and scores up to 100 validation images. If stopped,
rerunning **resumes** after the last finished epoch. It runs as a separate process so the notebook stays
usable; its log is `train_log.jsonl` in the output folder. Uses MPS here, CUDA on an NVIDIA machine.

Starting over: delete `segmenter.pt` first, or it resumes from it (and stops at once if its epochs are done).

In [ ]:
args = ["-m", "bila.vision.learned_segmenter", "train", "--data", TRAIN, "--val", VAL, "--out", OUT, "--epochs", EPOCHS]
if MAX_STEPS:
    args += ["--max-steps", MAX_STEPS]
run(*args)

In [ ]:
log = [json.loads(l) for l in open(OUT / "train_log.jsonl")]
loss_keys = [k for k in log[0] if k.startswith("loss_")]
fig, ax = plt.subplots(1, 2, figsize=(13, 3.5))
for k in loss_keys:
    ax[0].plot([r["epoch"] for r in log], [r[k] for r in log], "o-", label=k[5:])
ax[0].set(title="training losses, mean per epoch", xlabel="epoch"); ax[0].legend(fontsize=8)
if "recall" in log[0]:
    ax[1].plot([r["epoch"] for r in log], [r["recall"] for r in log], "o-", label="books found (IoU >= 0.5)")
    ax[1].plot([r["epoch"] for r in log], [r["iou"] for r in log], "o-", label="mean IoU of those found")
    ax[1].set(title="validation", xlabel="epoch", ylim=(0, 1)); ax[1].legend(fontsize=8)
plt.show()

## 4. Score it

Each true book is matched to the predicted mask that overlaps it most:
- **found**: the share of true books matched with IoU >= 0.5;
- **precision**: the share of predicted masks that are a book;
- **IoU of found**: how well the matched masks fit.

Rows are by kind of view and by where the book is. The robot's segmenter only ever sees **`sweep`** views (its
looks at the row: `sweep-0.3`, `sweep+0.0`, `sweep+0.3`); the close-ups (`close_row`, `close_tray`) are for the
sticker reader. So the rows that matter are `sweep` and **`target_row, >= 2000 px`**: the books the robot works
on, large enough to fit a spine to. The README's model (8 epochs on about 2900 images): 93% of those found,
mean IoU 0.93.


In [ ]:
from bila.vision import learned_segmenter as ls          # imports torch: no rendering in this kernel from here on
device = ls.default_device()
model = ls.load_model(str(OUT / "segmenter.pt"), device)
print("device:", device)
ls.print_scores(ls.evaluate(model, str(VAL), device, by=True))

## 5. Its masks next to the truth

Left: the model's masks. Right: the truth, books it missed outlined red. A book counts as found at IoU >= 0.5.

In [ ]:
seg = ls.LearnedSegmenter(str(OUT / "segmenter.pt"), device=device)
val = ls.BookImages(str(VAL))
for i in range(0, len(val), max(1, len(val) // 3))[:3]:
    stem, rgb, true = val.read(i)
    pred = seg.instances(rgb)
    found = {t for _, t, _ in ls.match(pred, true)}
    shown = rgb.copy()
    for k in range(1, int(true.max()) + 1):
        if k not in found and (true == k).sum() >= 500:      # missed books (not slivers at the image's edge)
            contours, _ = cv2.findContours((true == k).astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            cv2.drawContours(shown, contours, -1, (255, 0, 0), 3)
    fig, ax = plt.subplots(1, 2, figsize=(16, 4.5))
    ax[0].imshow((0.45 * rgb + 0.55 * colour_masks(pred, 1)).astype(np.uint8))
    ax[0].set_title(f"learned: {pred.max()} masks", fontsize=9)
    ax[1].imshow((0.45 * shown + 0.55 * colour_masks(true, 1)).astype(np.uint8))
    ax[1].set_title(f"truth: {true.max()} books, {len(found)} found, missed outlined red", fontsize=9)
    for a in ax: a.axis("off")
    fig.suptitle(stem, fontsize=10)
    plt.show()

## 6. Check it on the robot

The scores above are per pixel. What counts is whether the robot still shelves the book with the learned masks.
This runs the cart demo on a few scenes twice, with the simulator's masks and with the model, each in its own
process (rendering, and torch before it), all at once. About 3-5 minutes a run.

The scenes are ones the arm can reach in every variant (seed 2 of every scenario, and packed 0 and 1, fail
for reach whatever the camera does: `docs/calibration-and-insertion.md`).


In [ ]:
from concurrent.futures import ThreadPoolExecutor

CHECK = [("clear", 0), ("clear", 1), ("tight", 0), ("tight", 1), ("leaning", 0), ("packed", 2)]
WEIGHTS = OUT / "segmenter.pt"

def episode(scenario, seed, weights=None):
    args = [sys.executable, "-m", "bila.control.cart_demo", "--scenario", scenario, "--seed", str(seed)]
    if weights:
        args += ["--segmenter", str(weights)]
    out = subprocess.run(args, cwd=ROOT, capture_output=True, text=True).stdout.splitlines()
    sees = [l.strip() for l in out if "the camera sees" in l][:1]
    result = [l.strip() for l in out if l.strip().startswith(("SUCCESS", "FAIL", "SKIPPED"))][-1:]
    return (sees + result) or ["no result (crashed?)"]

jobs = [(s, k, w) for s, k in CHECK for w in (None, WEIGHTS)]
with ThreadPoolExecutor(max_workers=min(len(jobs), os.cpu_count() // 2)) as pool:
    results = list(pool.map(lambda j: episode(*j), jobs))
for (scenario, seed, w), lines in zip(jobs, results):
    print(f"{scenario:8s} {seed}  {'learned  ' if w else 'simulator'}  " + "\n                     ".join(lines))


A learned run that fails where the simulator's succeeds is worth a look in the live view, which tints every
mask the model found in its "last look" panel:

```bash
uv run python -m bila.viz.live_view --scenario tight --seed 1 --segmenter ~/bila_data/segmenter/segmenter.pt
```

## 7. Use it

From the repo root, with the weights this notebook wrote:

```bash
# the robot with the learned masks (headless; add --view under mjpython to watch it in MuJoCo)
uv run python -m bila.control.cart_demo --scenario tight --segmenter ~/bila_data/segmenter/segmenter.pt
# the same on a web page: wrist and scene cameras, the masks it found, the books it recognised
uv run python -m bila.viz.live_view --scenario tight --segmenter ~/bila_data/segmenter/segmenter.pt
# the first look's three views: learned masks, misses, IoUs, fitted spines
uv run python -m bila.viz.visualize --scenario leaning --seed 3 --learned --weights ~/bila_data
# the whole estimate against the truth, over many scenes (slow)
uv run python -m bila.vision.eval_perception collect --workers 3 --segmenter ~/bila_data/segmenter/segmenter.pt \
    --path ~/bila_data/perception_eval_learned.pkl
```

To improve it: more scenes and more epochs (the README's 670 / 8). The weak spot is books leaning on each
other, so adding leaning scenes helps most. If the robot fails where the simulator's masks succeed, look at the
`sweep` row of section 4 first: those are the only images the robot segments.
